<a href="https://colab.research.google.com/github/shanviMatrix/Assignment-ML-L/blob/main/Shanvi_2026010107_Assignment_LogReg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LAB Exercise: Logistic Regression
## Cancer Prediction using Logistic Regression

**Objective:** Use the cancer dataset (`data.csv`) to predict the outcome using Logistic Regression. Perform data cleaning, missing-value handling, correlation analysis, encoding and normalisation wherever required. Evaluate the model using Accuracy, Precision, Recall, F1-score and ROC curve.

**Note:** Keep `data.csv` in the same folder as this notebook. The notebook automatically looks for a common target column such as `diagnosis`, `target` or `class`. If your dataset uses a different target column, change `TARGET_COLUMN` in the setup cell.

## 1. Import required libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
print("Libraries imported successfully.")

## 2. Load the dataset

In [ ]:
# Keep data.csv in the same directory as this notebook
df = pd.read_csv("data.csv")

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())

## 3. Understand the dataset

In [ ]:
print("Dataset information:")
df.info()

print("\\nStatistical summary:")
display(df.describe(include="all").T)

print("\\nColumn names:")
print(df.columns.tolist())

## 4. Data cleaning

In [ ]:
# Remove duplicate rows
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()

# Remove common unnecessary index columns
unnecessary_cols = [
    col for col in df.columns
    if col.lower().startswith("unnamed") or col.lower() in ["id"]
]
df = df.drop(columns=unnecessary_cols, errors="ignore")

print("Shape after removing duplicates and unnecessary columns:", df.shape)

### 4.1 Check missing values

In [ ]:
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().mean() * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": missing_percentage
})
display(missing_summary)

plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, cmap="viridis")
plt.title("Missing Values in Dataset")
plt.show()

## 5. Select the target column

In [ ]:
# Automatically detect common target-column names.
# If detection is incorrect, enter the exact target column name below.
TARGET_COLUMN = None  # Example: "diagnosis"

if TARGET_COLUMN is None:
    possible_targets = ["diagnosis", "target", "class", "label", "outcome", "result"]
    matches = [col for col in df.columns if col.lower() in possible_targets]
    if matches:
        TARGET_COLUMN = matches[0]
    else:
        raise ValueError(
            "Target column was not detected. Set TARGET_COLUMN to the correct "
            "column name in this cell. Available columns: " + str(df.columns.tolist())
        )

print("Target column:", TARGET_COLUMN)
print("\\nTarget value counts:")
display(df[TARGET_COLUMN].value_counts(dropna=False))

## 6. Prepare the data

In [ ]:
# Remove rows where the target value is missing
df = df.dropna(subset=[TARGET_COLUMN]).copy()

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN].copy()

# Convert target labels into numeric values if they are categorical
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y.astype(str))

print("Target classes:", list(label_encoder.classes_))
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

## 7. Correlation analysis

In [ ]:
# Correlation between numerical columns
numeric_data = X.select_dtypes(include=np.number)

if numeric_data.shape[1] >= 2:
    plt.figure(figsize=(14, 10))
    correlation_matrix = numeric_data.corr()
    sns.heatmap(correlation_matrix, cmap="coolwarm", center=0, annot=False)
    plt.title("Correlation Heatmap of Numerical Features")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numerical columns for correlation heatmap.")

# Display highly correlated feature pairs
if numeric_data.shape[1] >= 2:
    corr = numeric_data.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    high_corr = upper.stack().sort_values(ascending=False)
    print("Highly correlated feature pairs (correlation > 0.90):")
    display(high_corr[high_corr > 0.90].to_frame("Correlation"))

## 8. Split the dataset into training and testing sets

We use an 80:20 split:
- 80% data for training.
- 20% data for testing.

`stratify=y` helps preserve the class distribution in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

## 9. Handle missing values, encode and normalise features

The preprocessing steps are placed inside a pipeline:
- Numerical columns: fill missing values with the median, then apply StandardScaler.
- Categorical columns: fill missing values with the most frequent value, then apply OneHotEncoder.

This ensures preprocessing is fitted only on the training data and helps avoid data leakage.

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

## 10. Train the Logistic Regression model

In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)
print("Logistic Regression model trained successfully.")

## 11. Make predictions

In [ ]:
y_pred = model.predict(X_test)

print("Actual values:   ", y_test[:10])
print("Predicted values:", y_pred[:10])

## 12. Evaluate classification metrics

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Score": [accuracy, precision, recall, f1]
})

display(metrics_df)
print("\\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    labels=np.arange(len(label_encoder.classes_)),
    target_names=[str(c) for c in label_encoder.classes_],
    zero_division=0
))

## 13. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Confusion Matrix")
plt.show()

## 14. ROC Curve and AUC Score

For binary classification, the ROC curve shows the relationship between the True Positive Rate and False Positive Rate at different thresholds. AUC measures the area under this curve.

In [ ]:
if len(label_encoder.classes_) == 2:
    y_probability = model.predict_proba(X_test)[:, 1]

    fpr, tpr, thresholds = roc_curve(y_test, y_probability)
    auc_score = roc_auc_score(y_test, y_probability)

    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {auc_score:.3f})")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.show()

    print("ROC-AUC Score:", round(auc_score, 4))
else:
    print("ROC curve in this notebook is implemented for binary classification.")
    print("For multiclass classification, use a One-vs-Rest ROC approach.")

## 15. Final conclusion

In [ ]:
print("FINAL MODEL PERFORMANCE")
print("-----------------------")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")

print("\\nConclusion:")
print("A Logistic Regression model was trained to predict the cancer dataset outcome.")
print("The metrics above describe its performance on the held-out test dataset.")
print("The ROC-AUC score, when available, measures the model's ability to distinguish")
print("between the two classes. Actual performance depends on the dataset and its quality.")

---
### Important notes
- The notebook expects a file named `data.csv` in the same folder.
- Check that the automatically selected target column is the correct outcome column before training.
- For a common breast-cancer dataset, `diagnosis` may contain classes such as `B` and `M`; the notebook encodes these labels automatically.
- The notebook reports weighted Precision, Recall and F1-score. If your lab specifically requires binary metrics for the positive class, change `average="weighted"` to `average="binary"` and set `pos_label` appropriately.
- This is a lab exercise, not a clinically validated cancer-diagnosis system.